In [ ]:
'''
spark - distributed processing engine

Apache Spark is a framework used to process large amounts of data.
PySpark is the Python API for Apache Spark.
Apache Spark is a framework used to process large amounts of data.
                Apache Spark
                    ↓
                Python API
                    ↓
                PySpark

-----------------------------------------------
                 Spark

              Large Data
                  ↓
        ┌─────────┼─────────┐
        ↓         ↓         ↓
     Machine   Machine   Machine
        ↓         ↓         ↓
     Process   Process   Process
        └─────────┼─────────┘
                  ↓
              Result


-sparksession -The object through which your Python program communicates with Spark.
-SparkSession.builder - This starts building/configuring your Spark session.
-.appName("MyFirstSparkApp") - Gives your Spark application a name.
-.getOrCreate() - Get an existing SparkSession if one exists; otherwise create a new one.
-spark is just a Python variable.It contains a SparkSession object.You then use it for Spark operations.

'''

In [ ]:
#basic pyspark program
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyFirstSparkApp") \
    .getOrCreate()

print("Spark started successfully")

spark.stop()

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()

# Create DataFrame
data = [
    ("Nalin", 21, "Chennai"),
    ("Rahul", 25, "Bangalore"),
    ("Arun", 30, "Mumbai")
]

columns = ["name", "age", "city"]

df = spark.createDataFrame(data, columns)

# Display
df.show()

# Schema
df.printSchema()

# Column names
df.columns

# Data types
df.dtypes

# Number of rows
df.count()

# Statistics
df.describe().show()

# Read CSV
df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

# Stop Spark
spark.stop()


In [ ]:
#schema describes the structure of your DataFrame.
df.printSchema()

'''
root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 '''

pyspark datatypes  

StringType
IntegerType
LongType
FloatType
DoubleType
BooleanType
DateType
TimestampType
ArrayType
StructType

In [ ]:
#PySpark DataFrame Operations

df.select("name", "age").show() #choose specific columns


from pyspark.sql.functions import col #refer to a column in a DataFrame using the col() function. This is useful when you want to perform operations on columns, such as arithmetic operations, filtering, or renaming.
df.select(
    col("name"),
    col("salary") * 2
).show()


df.filter(col("age") > 25).show() #filter rows based on a condition
df.filter(
    (col("age") > 25) &
    (col("salary") > 70000)
).show()



df.filter(
    ~(col("city") == "Chennai") #not condition
).show()

df = df.withColumn(     #create a new column in the DataFrame using the withColumn() method. This method takes two arguments: the name of the new column and the expression used to compute its values.
    "annual_salary",
    col("salary") * 12
)




In [ ]:
'''
Why do we write df =?

This is important.

You might think:

df.withColumn(...)

changes df.

But Spark DataFrames are immutable.

So we normally do:

df = df.withColumn(...)

Think:

Old DataFrame
      ↓
Transformation
      ↓
New DataFrame

The original DataFrame isn't directly modified.
'''

In [ ]:
df = df.withColumnRenamed(   #rename a column in the DataFrame using the withColumnRenamed() method
    "salary",
    "monthly_salary"
)

df.rename(
    columns={"salary": "monthly_salary"}
)


df = df.drop(  #drop columns from the DataFrame using the drop() method. This method takes one or more column names as arguments and returns a new DataFrame with the specified columns removed.
    "city",
    "age"
)

df.select(
    col("salary").alias("monthly_salary")  #alias for a column in the DataFrame using the alias() method. This method allows you to give a column a temporary name that can be used in subsequent operations.
).show()


from pyspark.sql.functions import lit
df = df.withColumn(
    "country",          
    lit("India")        #creates new column with a constant value using the lit() function. This function is used to create a column with a literal value, which can be useful for adding static information to your DataFrame.
)


df.distinct() #remove duplicate rows from the DataFrame using the distinct() method. This method returns a new DataFrame containing only unique rows, effectively eliminating any duplicates.

df.dropDuplicates(["city"]) #remove duplicate rows based on the values in the "city" column.


from pyspark.sql.functions import when
df = df.withColumn(
    "category",
    when(col("age") >= 30, "Senior")  #when() function is used to create conditional expressions in PySpark. It allows you to specify a condition and the corresponding value to assign when that condition is met. If the condition is not met, you can provide an alternative value using the otherwise() method.
    .otherwise("Junior")
)





String Functions

In [ ]:
from pyspark.sql.functions import upper

df.select(
    upper(col("name")).alias("name_upper")
).show()

In [ ]:
from pyspark.sql.functions import lower

df.select(
    lower(col("name")).alias("name_lower")
).show()

In [ ]:
from pyspark.sql.functions import trim

df.select(
    trim(col("name")).alias("clean_name")
).show()

In [ ]:
from pyspark.sql.functions import length

df.select(
    col("name"),
    length(col("name")).alias("name_length")
).show()

In [ ]:
from pyspark.sql.functions import substring

df.select(
    substring(col("name"), 1, 5).alias("first_five")
).show()

In [ ]:
from pyspark.sql.functions import concat

df.select(
    concat(
        col("first_name"),
        col("last_name")
    ).alias("full_name")
)

In [ ]:
from pyspark.sql.functions import split

df.select(
    split(col("name"), " ").alias("name_parts")
).show()

#["Nalin", "Karthik"]

Numerical functions

In [ ]:
data = [
    (1, 10.567),
    (2, 25.432),
    (3, -15.789),
    (4, 100.123)
]

df = spark.createDataFrame(
    data,
    ["id", "value"]
)

In [ ]:
from pyspark.sql.functions import round

df.select(
    round(col("value"), 2).alias("rounded")
).show()

In [ ]:
from pyspark.sql.functions import ceil

df.select(
    ceil(col("value"))
).show()

In [ ]:
from pyspark.sql.functions import floor

df.select(
    floor(col("value"))
).show()

In [ ]:
from pyspark.sql.functions import abs

df.select(
    abs(col("value")).alias("absolute_value")
).show()

In [ ]:
from pyspark.sql.functions import to_date

df = df.withColumn(
    "date",
    to_date(col("date_string"))
)

Null Handling


In [ ]:
data = [
    (1, "Nalin", 21),
    (2, None, 25),
    (3, "Arun", None),
    (4, None, None)
]

df = spark.createDataFrame(
    data,
    ["id", "name", "age"]
)

df.show()

In [ ]:
df.filter(
    col("name").isNull()   #Find rows where a column is NULL.
).show()

In [ ]:
df.filter(
    col("name").isNotNull() #Find rows where a column is NOT NULL.
).show()

In [ ]:
df = df.fillna({
    "name": "Unknown", #Fill missing values in the "name" column with "Unknown" and in the "age" column with 0 using the fillna() method. This method takes a dictionary where keys are column names and values are the values to fill in for missing data.
    "age": 0
})

In [ ]:
df.dropna()    #Drop rows with any NULL values

df.dropna(
    subset=["name"]    #Drop rows where the "name" column is NULL
)

Conditional functions

In [ ]:
from pyspark.sql.functions import when

df = df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 50000, "Medium")
    .otherwise("Low")
)

In [ ]:
'''
PySpark Functions
│
├── Column
│   └── col()
│
├── String
│   ├── upper()
│   ├── lower()
│   ├── trim()
│   ├── length()
│   ├── substring()
│   ├── split()
│   ├── concat()
│   └── regexp_replace()
│
├── Numeric
│   ├── round()
│   ├── ceil()
│   ├── floor()
│   ├── abs()
│   └── sqrt()
│
├── Date
│   ├── to_date()
│   ├── year()
│   ├── month()
│   ├── dayofmonth()
│   ├── datediff()
│   ├── date_add()
│   └── date_sub()
│
├── NULL
│   ├── isNull()
│   ├── isNotNull()
│   ├── fillna()
│   ├── dropna()
│   └── coalesce()
│
├── Conditional
│   ├── when()
│   └── otherwise()
│
└── Array
    ├── split()
    ├── size()
    └── array_contains()
    '''

In [ ]:
'''RDD - Resilient Distributed Dataset

	- RDD = Resilient Distributed Dataset
	- Distributed, immutable, fault-tolerant collection
	- Data divided into partitions and processed in parallel
	- Supports lazy evaluation and lineage
	- Accessed using SparkContext

	[RESILIENT]

	- Fault tolerant
	- Lost partitions recreated using lineage

	[DISTRIBUTED]

	- Data divided into partitions
	- Partitions processed in parallel

	[DATASET]

	- Collection of data
	- Can contain any Python object
    
    RDDs can still make sense when:

You need low-level control
You're working with unusual/unstructured objects
A specialized operation isn't convenient with DataFrames
You're maintaining older Spark code
You're studying Spark internals
'''

In [ ]:
numbers = [1, 2, 3, 4, 5]

rdd = spark.sparkContext.parallelize(numbers)

print(rdd.collect())

In [ ]:
#map()
result = rdd.map(lambda x: x * 2)
print(result.collect())


#filter()
result = rdd.filter(lambda x: x > 3)
print(result.collect())


#flatMap() - flattens the results of a function applied to each element in the RDD. It allows you to return multiple output elements for each input element, effectively "flattening" the results into a single RDD.
data = [
    "hello world",
    "hello spark"
]
rdd = spark.sparkContext.parallelize(data)
result = rdd.flatMap(lambda x: x.split(" "))
print(result.collect()) # ['hello', 'world', 'hello', 'spark']

#reduce() - combines the elements of the RDD using a specified binary function. It takes two elements at a time and applies the function to them, reducing the RDD to a single value. This is useful for aggregating data, such as summing numbers or finding the maximum value.
numbers = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(numbers)
result = rdd.reduce(lambda a, b: a + b)  
print(result) #15

#reduceByKey()
data = [
    ("IT", 50000),
    ("HR", 40000),
    ("IT", 60000),
    ("HR", 45000),
    ("IT", 70000)
]
rdd = spark.sparkContext.parallelize(data)
result = rdd.reduceByKey(
    lambda a, b: a + b
)
print(result.collect())  #[("IT", 180000), ("HR", 85000)]

#distinct()
numbers = [1, 2, 2, 3, 3, 3, 4]
rdd = spark.sparkContext.parallelize(numbers)
result = rdd.distinct()
print(result.collect())  #[1, 2, 3, 4]


#union()


#sortBy()
numbers = [5, 2, 8, 1, 3]
rdd = spark.sparkContext.parallelize(numbers)
result = rdd.sortBy(lambda x: x)
print(result.collect()) #[1, 2, 3, 5, 8]

In [ ]:
'''Creating RDD with partitions

You can specify the number of partitions:'''

rdd = spark.sparkContext.parallelize(
    [1, 2, 3, 4, 5, 6],
    3               #num of partitions
)


'''
RDD
 ↓
Low-level
 ↓
Collection of objects
 ↓
You control transformations
'''

aggregation

In [ ]:
# Number of rows
df.count()

# Average salary
df.select(
    avg("salary").alias("avg_salary")
).show()

# Minimum salary
df.select(
    min("salary").alias("min_salary")
).show()

# Maximum salary
df.select(
    max("salary").alias("max_salary")
).show()

# Number of non-null salaries
df.select(
    count("salary").alias("salary_count")
).show()

# Number of unique salaries
df.select(
    count_distinct("salary").alias("unique_salary_count")
).show()

# Total salary
df.agg(
    sum("salary").alias("total_salary")
).show()

# Total salary by department
df.groupBy("department").agg(
    sum("salary").alias("total_salary")
).show()

In [ ]:
df.agg(
    sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    count("salary").alias("count_salary"),
    count_distinct("salary").alias("distinct_salary_count")
).show()

Joins


In [ ]:
emp_df.join(
    dept_df,
    "dept_id",
    "inner"
)

In [ ]:
result = emp_df.join(
    dept_df,
    emp_df.department_id == dept_df.dept_id,   #columns names are different but have same values
    "inner"
)

In [ ]:

# An inner join returns only rows that have a match in both DataFrames.

result = emp_df.join(
    dept_df,
    "dept_id",
    "inner"
)

result.show()

In [ ]:
#left join keeps all rows from the left DataFrame.
result = emp_df.join(
    dept_df,
    "dept_id",
    "left"
)
result.show()

In [ ]:
#right join keeps all rows from the right DataFrame.
result = emp_df.join(
    dept_df,
    "dept_id",
    "right"
)
result.show()

In [ ]:
#Full join keeps everything from both DataFrames.
result = emp_df.join(
    dept_df,
    "dept_id",
    "full"
)
result.show()

In [ ]:
#cross join creates every possible combination of rows.
result = emp_df.join(
    dept_df,
    how="cross"
)
result.show()

In [ ]:
#left semi join returns rows from the left DataFrame where a match exists in the right DataFrame.
result = emp_df.join(
    dept_df,
    "dept_id",
    "left_semi"
)

In [ ]:
#Left anti is the opposite, It returns rows from the left DataFrame where no match exists in the right
result = emp_df.join(
    dept_df,
    "dept_id",
    "left_anti"
)
result.show()

In [ ]:
'''
Broadcast Join in PySpark

Broadcast join is a performance optimization for joins.

The basic idea is very simple:

If one DataFrame is small enough, Spark can copy it to every executor instead of shuffling the large DataFrame around.
A broadcast join is a Spark optimization used when one side of a join is small enough to fit comfortably in executor memory. 
Spark broadcasts the small DataFrame to the executors, 
allowing the large DataFrame partitions to join against the local copy and reducing expensive shuffle operations.
'''

In [ ]:
result = emp_df.join(
    broadcast(dept_df),
    "dept_id",
    "inner"
)

In [ ]:
'''
Handling Duplicate Columns

There are actually two different meanings of "duplicate columns":

1 Same column appears in both DataFrames after a join.
2 The DataFrame itself has duplicate column names.
'''

In [ ]:
# ============================================================
# PYSPARK: HANDLING DUPLICATE COLUMNS AND NULL VALUES
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    when,
    avg,
    sum,
    coalesce
)


# ============================================================
# PART 1: HANDLING DUPLICATE COLUMNS AFTER A JOIN
# ============================================================

# Suppose we have two DataFrames:
#
# emp_df:
# employee_id | name  | department_id | salary
#
# dept_df:
# dept_id | dept_name
#
# Notice:
# emp_df       -> department_id
# dept_df      -> dept_id
#
# The join columns have DIFFERENT names.


# ------------------------------------------------------------
# 1. JOIN WHEN BOTH DATAFRAMES HAVE THE SAME COLUMN NAME
# ------------------------------------------------------------

# If both DataFrames have a column called "dept_id",
# we can use the short syntax:
#
# df1.join(df2, "dept_id", "inner")
#
# Spark knows that "dept_id" is the join column
# and keeps only one copy of the join column.

result = emp_df.join(
    dept_df,
    "dept_id",
    "inner"
)


# ------------------------------------------------------------
# 2. JOIN WHEN COLUMN NAMES ARE DIFFERENT
# ------------------------------------------------------------

# If emp_df has "department_id"
# and dept_df has "dept_id",
# we cannot use:
#
# emp_df.join(dept_df, "dept_id", "inner")
#
# because emp_df does not contain "dept_id".
#
# Instead, explicitly specify the join condition.

result = emp_df.join(
    dept_df,
    emp_df.department_id == dept_df.dept_id,
    "inner"
)


# ------------------------------------------------------------
# 3. WHY DUPLICATE COLUMNS CAN BE A PROBLEM
# ------------------------------------------------------------

# After the above join, we may have both:
#
# department_id
# dept_id
#
# If both DataFrames had a column with the same name,
# we could also end up with duplicate column names.
#
# Trying to refer to an ambiguous column like:
#
# result.select("dept_id")
#
# can cause:
#
# AnalysisException: Reference 'dept_id' is ambiguous
#
# Spark does not know which column you mean.


# ------------------------------------------------------------
# 4. USE ALIAS TO HANDLE DUPLICATE/AMBIGUOUS COLUMNS
# ------------------------------------------------------------

# alias() gives each DataFrame a temporary name.

emp = emp_df.alias("e")
dept = dept_df.alias("d")

result = emp.join(
    dept,
    col("e.department_id") == col("d.dept_id"),
    "inner"
)


# Now we can clearly specify where each column comes from:
#
# e.department_id -> department_id from employee DataFrame
# d.dept_id       -> dept_id from department DataFrame
#
# This avoids ambiguity.


# ------------------------------------------------------------
# 5. SELECT ONLY THE COLUMNS WE NEED AFTER A JOIN
# ------------------------------------------------------------

# It is usually better not to keep every column from both
# DataFrames after a join.
#
# Explicitly select the columns required in the final result.

result = (
    emp
    .join(
        dept,
        col("e.department_id") == col("d.dept_id"),
        "inner"
    )
    .select(
        col("e.employee_id"),
        col("e.name"),
        col("e.salary"),
        col("d.dept_name")
    )
)

# Result:
#
# employee_id | name  | salary | dept_name
# ------------------------------------------
# 1           | Nalin | 50000  | IT
# 2           | Rahul | 60000  | HR


# ------------------------------------------------------------
# 6. RENAME A COLUMN USING alias()
# ------------------------------------------------------------

# Sometimes we actually need both columns.
# In that case, rename them to avoid duplicate names.

result = (
    emp
    .join(
        dept,
        col("e.department_id") == col("d.dept_id"),
        "inner"
    )
    .select(
        col("e.department_id").alias("employee_dept_id"),
        col("d.dept_id").alias("department_dept_id"),
        col("e.name"),
        col("d.dept_name")
    )
)

# Result:
#
# employee_dept_id | department_dept_id | name  | dept_name
# -----------------------------------------------------------
# 10               | 10                 | Nalin | IT
# 20               | 20                 | Rahul | HR

In [ ]:
# ============================================================
# PART 2: HANDLING NULL VALUES
# ============================================================

# NULL means the value is missing or unknown.
#
# Example:
#
# salary
# -------
# 50000
# NULL
# 60000


# ------------------------------------------------------------
# 7. CHECK FOR NULL VALUES
# ------------------------------------------------------------

# isNull() returns True when the column value is NULL.

df.filter(
    col("salary").isNull()
).show()

# This returns only rows where salary is NULL.


# ------------------------------------------------------------
# 8. CHECK FOR NON-NULL VALUES
# ------------------------------------------------------------

# isNotNull() returns True when the value is NOT NULL.

df.filter(
    col("salary").isNotNull()
).show()

# This returns only rows where salary contains a value.


# ------------------------------------------------------------
# 9. REMOVE ROWS CONTAINING NULL
# ------------------------------------------------------------

# dropna() removes rows that contain NULL values.

df_clean = df.dropna()

# Example:
#
# Before:
#
# Nalin   50000
# Rahul   NULL
# Arun    60000
#
# After dropna():
#
# Nalin   50000
# Arun    60000


# ------------------------------------------------------------
# 10. REMOVE ROWS WHERE A SPECIFIC COLUMN IS NULL
# ------------------------------------------------------------

# Sometimes we only care about NULL values in one column.

df_clean = df.dropna(
    subset=["salary"]
)

# This removes rows where salary is NULL,
# but does not necessarily remove rows because
# another column contains NULL.


# ------------------------------------------------------------
# 11. FILL NULL VALUES WITH A FIXED VALUE
# ------------------------------------------------------------

# fillna() replaces NULL values with a specified value.

df_clean = df.fillna({
    "salary": 0
})

# Example:
#
# Before:
#
# salary
# -------
# 50000
# NULL
# 60000
#
# After:
#
# salary
# -------
# 50000
# 0
# 60000


# ------------------------------------------------------------
# 12. FILL NULL VALUES IN MULTIPLE COLUMNS
# ------------------------------------------------------------

df_clean = df.fillna({
    "salary": 0,
    "city": "Unknown",
    "age": 0
})

# Each column can have its own replacement value.


# ------------------------------------------------------------
# 13. FILL NULL WITH THE AVERAGE
# ------------------------------------------------------------

# Instead of replacing a missing salary with 0,
# we can replace it with the average salary.

avg_salary = df.select(
    avg("salary")
).first()[0]

df_clean = df.fillna({
    "salary": avg_salary
})

# .first()[0] gets the actual average value from the
# one-row DataFrame returned by select(avg("salary")).


# ============================================================
# PART 3: COALESCE
# ============================================================

# coalesce() returns the FIRST NON-NULL value.
#
# Example:
#
# coalesce(phone, alternate_phone, "Not Available")
#
# Meaning:
#
# 1. If phone exists -> use phone
# 2. If phone is NULL -> use alternate_phone
# 3. If both are NULL -> use "Not Available"


df = df.withColumn(
    "final_phone",
    coalesce(
        col("phone"),
        col("alternate_phone"),
        lit("Not Available")
    )
)


# ------------------------------------------------------------
# 14. SIMPLE COALESCE EXAMPLE
# ------------------------------------------------------------

# If city is NULL, use "Unknown".

df = df.withColumn(
    "final_city",
    coalesce(
        col("city"),
        lit("Unknown")
    )
)

# Example:
#
# city
# -------
# Chennai
# NULL
# Mumbai
#
# final_city
# ----------
# Chennai
# Unknown
# Mumbai


# ============================================================
# PART 4: NULL + WHEN()
# ============================================================

# We can combine isNull(), when(), and otherwise()
# to create a status column.

df = df.withColumn(
    "salary_status",
    when(
        col("salary").isNull(),
        "Missing"
    ).otherwise(
        "Available"
    )
)

# Result:
#
# salary | salary_status
# ----------------------
# 50000  | Available
# NULL   | Missing
# 60000  | Available


# ============================================================
# PART 5: NULL AFTER A LEFT JOIN
# ============================================================

# NULL values commonly appear after a LEFT JOIN.
#
# Example:
#
# Employees:
#
# name   dept_id
# Nalin  10
# Rahul  20
# Priya  40
#
# Departments:
#
# dept_id   dept_name
# 10        IT
# 20        HR
#
# Priya has dept_id = 40, but there is no department 40.
#
# Therefore, after a LEFT JOIN:
#
# name   dept_id   dept_name
# Nalin  10        IT
# Rahul  20        HR
# Priya  40        NULL


emp = emp_df.alias("e")
dept = dept_df.alias("d")

result = (
    emp
    .join(
        dept,
        col("e.dept_id") == col("d.dept_id"),
        "left"
    )
    .select(
        col("e.employee_id"),
        col("e.name"),
        col("e.dept_id"),
        col("d.dept_name")
    )
)


# ------------------------------------------------------------
# 15. HANDLE NULLS CREATED BY THE LEFT JOIN
# ------------------------------------------------------------

# Replace NULL department names with "Unknown".

result = result.fillna({
    "dept_name": "Unknown"
})

# Final result:
#
# employee_id | name  | dept_id | dept_name
# -------------------------------------------
# 1           | Nalin | 10      | IT
# 2           | Rahul | 20      | HR
# 3           | Priya | 40      | Unknown


# ============================================================
# PART 6: NULL VALUES AND AGGREGATIONS
# ============================================================

# Spark aggregation functions generally ignore NULL values.
#
# Example:
#
# salary
# -------
# 50000
# 60000
# NULL
# 70000
#
# avg("salary") calculates:
#
# (50000 + 60000 + 70000) / 3
#
# It does NOT treat NULL as another value.


# avg() ignores NULL
df.select(
    avg("salary")
).show()


# count("salary") counts NON-NULL salary values.
df.select(
    count("salary")
).show()


# count("*") counts ALL rows, including rows where salary is NULL.
df.select(
    count("*")
).show()


# ============================================================
# PART 7: NULL VS EMPTY STRING
# ============================================================

# NULL and an empty string "" are NOT the same.
#
# NULL:
# -> value is missing/unknown
#
# "":
# -> a string exists, but it contains zero characters.


# Check NULL:
df.filter(
    col("name").isNull()
).show()


# Check empty string:
df.filter(
    col("name") == ""
).show()


# Check BOTH NULL and empty string:
df.filter(
    col("name").isNull() |
    (col("name") == "")
).show()


In [ ]:
# ============================================================
# PYSPARK: PARQUET AND ORC
# ============================================================

# Parquet and ORC are file formats used to store large datasets.
#
# Both are:
# - Columnar storage formats
# - Designed for big-data processing
# - Compressed
# - Schema-aware
# - Well supported by PySpark
#
# They are generally much better suited for large analytical
# workloads than CSV files.


# ============================================================
# PART 1: CSV vs PARQUET / ORC
# ============================================================

# CSV:
#
# - Text-based file format
# - Human-readable
# - Usually larger
# - Does not efficiently store data types/schema
# - Good for simple data exchange
#
#
# Parquet / ORC:
#
# - Binary columnar formats
# - Store data by columns
# - Support compression
# - Store schema information
# - Efficient for analytical queries
# - Commonly used in data lakes and Spark pipelines


# ============================================================
# PART 2: WHAT DOES COLUMNAR STORAGE MEAN?
# ============================================================

# Suppose the data is:
#
# id    name    age    salary
# 1     Nalin   21     50000
# 2     Rahul   25     60000
# 3     Arun    30     75000
#
#
# Row-oriented storage conceptually stores:
#
# 1, Nalin, 21, 50000
# 2, Rahul, 25, 60000
# 3, Arun, 30, 75000
#
#
# Columnar storage conceptually organizes data as:
#
# id:
# 1
# 2
# 3
#
# name:
# Nalin
# Rahul
# Arun
#
# age:
# 21
# 25
# 30
#
# salary:
# 50000
# 60000
# 75000
#
# This is useful for analytical queries because Spark can
# avoid reading columns that are not needed.


# ============================================================
# PART 3: PARQUET
# ============================================================

# Parquet is a columnar storage format widely used in
# Spark and modern data-engineering systems.
#
# It is one of the MOST IMPORTANT file formats to know
# for PySpark.


# ------------------------------------------------------------
# 1. WRITE A DATAFRAME TO PARQUET
# ------------------------------------------------------------

# Suppose:
#
# df
#
# is our PySpark DataFrame.

df.write.parquet("employees")


# Spark normally creates a DIRECTORY rather than one
# single .parquet file.
#
# Conceptually:
#
# employees/
#     part-00000-....parquet
#     part-00001-....parquet
#     part-00002-....parquet
#     _SUCCESS
#
# Multiple files are created because Spark processes
# data using partitions.


# ------------------------------------------------------------
# 2. WRITE PARQUET WITH OVERWRITE
# ------------------------------------------------------------

# By default, Spark does not overwrite an existing destination.
#
# mode("overwrite") replaces the existing output.

df.write \
    .mode("overwrite") \
    .parquet("employees")


# ------------------------------------------------------------
# 3. READ PARQUET
# ------------------------------------------------------------

# Reading Parquet is very simple.

new_df = spark.read.parquet(
    "employees"
)

new_df.show()

# Parquet stores schema information, so Spark can recover
# the schema when reading the data.


# ------------------------------------------------------------
# 4. CHECK THE SCHEMA
# ------------------------------------------------------------

new_df.printSchema()

# Example:
#
# root
#  |-- id: long
#  |-- name: string
#  |-- age: long
#  |-- salary: long
#
# Unlike CSV, you normally do not need:
#
# header=True
# inferSchema=True
#
# because Parquet stores schema information.


# ============================================================
# PART 4: WRITE MODES
# ============================================================

# overwrite
# -> Replace existing data.

df.write \
    .mode("overwrite") \
    .parquet("employees")


# append
# -> Add new data to existing data.

df.write \
    .mode("append") \
    .parquet("employees")


# ignore
# -> If the destination already exists, do nothing.

df.write \
    .mode("ignore") \
    .parquet("employees")


# error / errorifexists
# -> Throw an error if the destination already exists.
# -> This is the default behavior.

df.write \
    .mode("error") \
    .parquet("employees")


# ============================================================
# PART 5: COMPRESSION
# ============================================================

# Parquet supports compression.
#
# Compression reduces storage size and can reduce the amount
# of data that needs to be read from storage.
#
# Common compression codecs:
#
# - Snappy
# - Gzip
# - Zstandard
#
# Snappy is a common choice in Spark because it provides
# a good balance between compression and speed.


df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("employees")


# ============================================================
# PART 6: PARTITIONED PARQUET
# ============================================================

# Partitioning is extremely important in PySpark.
#
# Suppose we have:
#
# customer_id
# region
# year
# month
# amount
#
# We can partition the output by year and month.

df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("sales")


# Spark creates a directory structure similar to:
#
# sales/
# ├── year=2025/
# │   ├── month=01/
# │   ├── month=02/
# │   └── month=03/
# │
# └── year=2026/
#     ├── month=01/
#     ├── month=02/
#     └── month=03/
#
#
# Each partition contains Parquet files.


# ============================================================
# PART 7: PARTITION PRUNING
# ============================================================

# Suppose the data is partitioned by year:
#
# sales/
# ├── year=2024/
# ├── year=2025/
# └── year=2026/
#
#
# Now we query only 2026:

result = df.filter(
    col("year") == 2026
)


# Spark can use the partition information to avoid reading
# unrelated partitions such as:
#
# year=2024
# year=2025
#
# and read only:
#
# year=2026
#
# This is called PARTITION PRUNING.
#
# Partition pruning can significantly improve performance
# when the partition column is used in filters.


# ============================================================
# PART 8: ORC
# ============================================================

# ORC stands for:
#
# Optimized Row Columnar
#
# Despite the name, ORC is a COLUMNAR storage format.
#
# ORC is commonly associated with the Hadoop/Hive ecosystem
# and is also well supported by Spark.


# ------------------------------------------------------------
# 1. WRITE ORC
# ------------------------------------------------------------

df.write.orc(
    "employees_orc"
)


# ------------------------------------------------------------
# 2. WRITE ORC WITH OVERWRITE
# ------------------------------------------------------------

df.write \
    .mode("overwrite") \
    .orc("employees_orc")


# ------------------------------------------------------------
# 3. READ ORC
# ------------------------------------------------------------

orc_df = spark.read.orc(
    "employees_orc"
)

orc_df.show()


# ------------------------------------------------------------
# 4. CHECK ORC SCHEMA
# ------------------------------------------------------------

orc_df.printSchema()


# ============================================================
# PART 9: ORC WITH COMPRESSION
# ============================================================

# ORC also supports compression.

df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .orc("employees_orc")


# ============================================================
# PART 10: PARTITIONED ORC
# ============================================================

# ORC can also be partitioned.

df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .orc("sales_orc")


# The directory structure is conceptually:
#
# sales_orc/
# ├── year=2025/
# │   ├── month=01/
# │   └── month=02/
# │
# └── year=2026/
#     ├── month=01/
#     └── month=02/


# ============================================================
# PART 11: PARQUET VS ORC
# ============================================================

# PARQUET:
#
# - Columnar format
# - Compressed
# - Stores schema
# - Excellent Spark support
# - Very widely used across data-engineering tools
# - Very common in data lakes
#
#
# ORC:
#
# - Columnar format
# - Compressed
# - Stores schema
# - Excellent Spark/Hive support
# - Strong association with Hadoop/Hive ecosystems
#
#
# Both are designed for efficient analytical processing.


# ============================================================
# PART 12: PARQUET VS CSV
# ============================================================

# CSV:
#
# Data
# ↓
# Text
# ↓
# Parse data
# ↓
# Infer/define types
#
#
# Parquet:
#
# Data
# ↓
# Columnar binary format
# ↓
# Schema stored
# ↓
# Efficient reading
#
#
# Parquet is generally preferred for internal Spark
# data pipelines, while CSV is often useful for
# exchanging data with external systems.


# ============================================================
# PART 13: COMPLETE PARQUET EXAMPLE
# ============================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ParquetExample") \
    .getOrCreate()


data = [
    (1, "Nalin", 21, 50000),
    (2, "Rahul", 25, 60000),
    (3, "Arun", 30, 75000)
]

columns = [
    "id",
    "name",
    "age",
    "salary"
]

df = spark.createDataFrame(
    data,
    columns
)


# Write DataFrame to Parquet

df.write \
    .mode("overwrite") \
    .parquet("employees_parquet")


# Read Parquet

parquet_df = spark.read.parquet(
    "employees_parquet"
)


# Display data

parquet_df.show()


# Display schema

parquet_df.printSchema()


# ============================================================
# PART 14: COMPLETE ORC EXAMPLE
# ============================================================

# Write DataFrame to ORC

df.write \
    .mode("overwrite") \
    .orc("employees_orc")


# Read ORC

orc_df = spark.read.orc(
    "employees_orc"
)


# Display data

orc_df.show()


# Display schema

orc_df.printSchema()


# ============================================================
# PART 15: IMPORTANT MENTAL MODEL
# ============================================================

# CSV
#
#     Text-based
#         ↓
#     Simple
#         ↓
#     Usually not ideal for large analytical processing
#
#
# PARQUET
#
#     Columnar
#         ↓
#     Compressed
#         ↓
#     Schema stored
#         ↓
#     Efficient analytical queries
#         ↓
#     VERY COMMON WITH SPARK
#
#
# ORC
#
#     Columnar
#         ↓
#     Compressed
#         ↓
#     Schema stored
#         ↓
#     Efficient analytical queries
#         ↓
#     Common in Hadoop/Hive environments


# ============================================================
# QUICK CHEAT SHEET
# ============================================================

# ---------- PARQUET ----------

# Write:
df.write.parquet("path")

# Write with overwrite:
df.write.mode("overwrite").parquet("path")

# Read:
df = spark.read.parquet("path")

# Compression:
df.write \
    .option("compression", "snappy") \
    .parquet("path")

# Partition:
df.write \
    .partitionBy("year", "month") \
    .parquet("path")


# ---------- ORC ----------

# Write:
df.write.orc("path")

# Write with overwrite:
df.write.mode("overwrite").orc("path")

# Read:
df = spark.read.orc("path")

# Compression:
df.write \
    .option("compression", "snappy") \
    .orc("path")

# Partition:
df.write \
    .partitionBy("year", "month") \
    .orc("path")


# ============================================================
# MOST IMPORTANT THINGS TO REMEMBER
# ============================================================

# 1. Parquet and ORC are COLUMNAR file formats.
#
# 2. Both support compression and store schema information.
#
# 3. Parquet is extremely common in modern Spark/data-lake
#    environments.
#
# 4. Spark usually writes multiple Parquet/ORC files because
#    the DataFrame is divided into partitions.
#
# 5. partitionBy() creates directory-level partitions.
#
# 6. Partition pruning allows Spark to skip partitions that
#    do not satisfy a filter.
#
# 7. For modern PySpark, prioritize learning PARQUET first.
#
# 8. Remember these four basic operations:
#
#    spark.read.parquet("path")
#    df.write.parquet("path")
#
#    spark.read.orc("path")
#    df.write.orc("path")

In [ ]:
'''
Common Table Expression (CTE)

Since you know MySQL, CTE will be very familiar. The important thing for PySpark is understanding CTE in Spark SQL.

A CTE is a temporary named result that you create inside a SQL query and then use in the main query.

Think of it as:

Complex query
     ↓
Break into smaller query
     ↓
Give that result a name
     ↓
Use that name in the next query

DataFrame
    ↓
createOrReplaceTempView()
    ↓
SQL table/view
    ↓
WITH
    ↓
CTE
    ↓
temporary named query result
    ↓
Main SELECT
    ↓
Final DataFrame
'''

In [ ]:
Without a CTE, the query can become harder to read:

SELECT *
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);

With a CTE:

WITH avg_salary AS (
    SELECT AVG(salary) AS avg_sal
    FROM employees
)
SELECT *
FROM employees
CROSS JOIN avg_salary
WHERE employees.salary > avg_salary.avg_sal;

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CTEExample") \
    .getOrCreate()

data = [
    (1, "Nalin", "IT", 50000),
    (2, "Rahul", "IT", 70000),
    (3, "Arun", "HR", 45000),
    (4, "Priya", "HR", 60000)
]

columns = [
    "id",
    "name",
    "department",
    "salary"
]

df = spark.createDataFrame(
    data,
    columns
)

# Register DataFrame as a temporary SQL view
df.createOrReplaceTempView("employees")

# Use a CTE
result = spark.sql("""
    WITH high_salary AS (
        SELECT *
        FROM employees
        WHERE salary > 50000
    )

    SELECT *
    FROM high_salary
""")

result.show()

In [ ]:
# ============================================================
# PYSPARK: USER DEFINED FUNCTIONS (UDF)
# ============================================================

# UDF stands for User Defined Function.
#
# A UDF allows us to create our own Python function and use
# that function on Spark DataFrame columns.
#
# We use UDF when the required logic cannot be easily achieved
# using Spark's built-in functions.
#
# IMPORTANT:
# Spark has many built-in functions such as:
#
# upper()
# lower()
# trim()
# substring()
# when()
# regexp_replace()
# etc.
#
# Always prefer built-in Spark functions when possible because
# they are generally more efficient than Python UDFs.


# ============================================================
# PART 1: NORMAL PYTHON FUNCTION
# ============================================================

# Suppose we have a normal Python function:

def get_grade(marks):
    if marks >= 90:
        return "A"
    elif marks >= 75:
        return "B"
    elif marks >= 60:
        return "C"
    else:
        return "D"


# This works normally in Python:

print(get_grade(85))

# Output:
# B


# ============================================================
# PART 2: WHY CAN'T WE DIRECTLY USE IT ON A DATAFRAME COLUMN?
# ============================================================

# Suppose our DataFrame has:
#
# name    marks
# Nalin   85
# Rahul   72
# Arun    55
#
# We cannot normally do:
#
# df.withColumn(
#     "grade",
#     get_grade(col("marks"))
# )
#
# because get_grade() is an ordinary Python function.
#
# Spark needs to know that this Python function should be
# applied to values in a Spark column.


# ============================================================
# PART 3: CREATE A UDF
# ============================================================

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


# Convert the Python function into a Spark UDF.

grade_udf = udf(
    get_grade,
    StringType()
)

# The second argument tells Spark the RETURN TYPE
# of the function.
#
# get_grade() returns:
#
# "A", "B", "C", "D"
#
# Therefore the return type is:
#
# StringType()


# ============================================================
# PART 4: USE THE UDF WITH withColumn()
# ============================================================

df = df.withColumn(
    "grade",
    grade_udf(col("marks"))
)

# Example:
#
# Before:
#
# +-----+-----+
# | name|marks|
# +-----+-----+
# |Nalin|   85|
# |Rahul|   72|
# | Arun|   55|
# +-----+-----+
#
#
# After:
#
# +-----+-----+-----+
# | name|marks|grade|
# +-----+-----+-----+
# |Nalin|   85|    B|
# |Rahul|   72|    C|
# | Arun|   55|    D|
# +-----+-----+-----+


# ============================================================
# PART 5: COMPLETE UDF EXAMPLE
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

spark = SparkSession.builder \
    .appName("UDFExample") \
    .getOrCreate()


data = [
    ("Nalin", 85),
    ("Rahul", 72),
    ("Arun", 55),
    ("Priya", 95)
]

columns = [
    "name",
    "marks"
]

df = spark.createDataFrame(
    data,
    columns
)


def get_grade(marks):

    if marks >= 90:
        return "A"

    elif marks >= 75:
        return "B"

    elif marks >= 60:
        return "C"

    else:
        return "D"


# Register the Python function as a Spark UDF.

grade_udf = udf(
    get_grade,
    StringType()
)


# Apply the UDF to the marks column.

result = df.withColumn(
    "grade",
    grade_udf(col("marks"))
)

result.show()


# ============================================================
# PART 6: UDF WITH INTEGER RETURN TYPE
# ============================================================

# The return type of a UDF must match the type returned
# by the Python function.


def add_bonus(salary):
    return salary + 5000


# The function returns a number,
# so we use IntegerType().

from pyspark.sql.types import IntegerType

bonus_udf = udf(
    add_bonus,
    IntegerType()
)


result = df.withColumn(
    "new_salary",
    bonus_udf(col("salary"))
)


# ============================================================
# PART 7: COMMON PYSPARK UDF RETURN TYPES
# ============================================================

from pyspark.sql.types import (
    StringType,
    IntegerType,
    DoubleType,
    FloatType,
    BooleanType
)


# String return:
#
# StringType()


# Integer return:
#
# IntegerType()


# Decimal / floating-point return:
#
# DoubleType()


# Boolean return:
#
# BooleanType()


# Example:

def is_adult(age):
    return age >= 18

adult_udf = udf(
    is_adult,
    BooleanType()
)


# ============================================================
# PART 8: UDF WITH MULTIPLE COLUMNS
# ============================================================

# A UDF can accept multiple DataFrame columns.


def calculate_total(price, quantity):
    return price * quantity


total_udf = udf(
    calculate_total,
    DoubleType()
)


result = df.withColumn(
    "total",
    total_udf(
        col("price"),
        col("quantity")
    )
)


# The function receives:
#
# price
#   +
# quantity
#   ↓
# calculate_total()
#   ↓
# total


# ============================================================
# PART 9: UDF USING lambda
# ============================================================

# For very simple functions, we can use lambda.

upper_udf = udf(
    lambda x: x.upper(),
    StringType()
)


result = df.withColumn(
    "upper_name",
    upper_udf(col("name"))
)


# However, for complex logic, defining a normal function
# is usually easier to read.


# ============================================================
# PART 10: HANDLING NULL VALUES IN UDF
# ============================================================

# UDFs can receive NULL values.
#
# Python represents NULL as None.


def safe_grade(marks):

    if marks is None:
        return "Unknown"

    if marks >= 90:
        return "A"

    elif marks >= 75:
        return "B"

    elif marks >= 60:
        return "C"

    else:
        return "D"


safe_grade_udf = udf(
    safe_grade,
    StringType()
)


result = df.withColumn(
    "grade",
    safe_grade_udf(col("marks"))
)


# IMPORTANT:
#
# If the column can contain NULL values,
# handle None inside the Python function when necessary.


# ============================================================
# PART 11: UDF vs BUILT-IN FUNCTION
# ============================================================

# Suppose we want to convert names to uppercase.
#
# We COULD create a UDF:

def make_upper(name):
    return name.upper()

upper_udf = udf(
    make_upper,
    StringType()
)

df.withColumn(
    "upper_name",
    upper_udf(col("name"))
)


# BUT THIS IS NOT RECOMMENDED.
#
# Spark already has a built-in function:

from pyspark.sql.functions import upper

df.withColumn(
    "upper_name",
    upper(col("name"))
)


# Prefer:
#
# upper(col("name"))
#
# instead of:
#
# upper_udf(col("name"))
#
# whenever a built-in Spark function can perform the task.


# ============================================================
# PART 12: WHY UDFs CAN BE SLOW
# ============================================================

# Normal Spark built-in function:
#
# DataFrame
#     ↓
# Spark built-in function
#     ↓
# Spark execution engine
#
#
# Python UDF:
#
# DataFrame
#     ↓
# Spark
#     ↓
# Python function
#     ↓
# Spark
#
#
# Moving between Spark's execution environment and Python
# can introduce additional overhead.
#
# Therefore Python UDFs can be significantly slower than
# equivalent built-in Spark functions, especially on large
# datasets.


# ============================================================
# PART 13: BUILT-IN FUNCTION vs UDF EXAMPLE
# ============================================================

# Suppose we want to convert names to lowercase.
#
# BAD / unnecessary UDF:

def make_lower(name):
    return name.lower()

lower_udf = udf(
    make_lower,
    StringType()
)

df.withColumn(
    "lower_name",
    lower_udf(col("name"))
)


# BETTER:
#
# Use Spark's built-in lower():

from pyspark.sql.functions import lower

df.withColumn(
    "lower_name",
    lower(col("name"))
)


# ============================================================
# PART 14: WHEN SHOULD WE USE A UDF?
# ============================================================

# Use a UDF when:
#
# 1. You have custom business logic.
#
# 2. Spark's built-in functions cannot easily perform
#    the required operation.
#
# 3. You need to apply a custom Python function to
#    DataFrame column values.
#
#
# Example:
#
# A company has a custom scoring rule:
#
# score < 20     -> "Low"
# 20-50          -> "Medium"
# > 50           -> "High"
#
# If the logic is complicated enough or doesn't map cleanly
# to built-in functions, a UDF may be appropriate.


# ============================================================
# PART 15: UDF MENTAL MODEL
# ============================================================

# Normal Python function:
#
# Python value
#      ↓
# Python function
#      ↓
# Python value
#
#
# UDF:
#
# Spark Column
#      ↓
# Spark UDF
#      ↓
# Python function
#      ↓
# Result Column


# ============================================================
# PART 16: UDF SYNTAX TO MEMORIZE
# ============================================================

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


def my_function(value):
    # custom logic
    return ...


my_udf = udf(
    my_function,
    StringType()
)


df = df.withColumn(
    "new_column",
    my_udf(col("existing_column"))
)


# ============================================================
# QUICK CHEAT SHEET
# ============================================================

# Create normal Python function:

def get_grade(marks):
    if marks >= 90:
        return "A"
    elif marks >= 75:
        return "B"
    else:
        return "C"


# Convert Python function to UDF:

grade_udf = udf(
    get_grade,
    StringType()
)


# Apply UDF:

df = df.withColumn(
    "grade",
    grade_udf(col("marks"))
)


# UDF with multiple columns:

total_udf = udf(
    calculate_total,
    DoubleType()
)

df = df.withColumn(
    "total",
    total_udf(
        col("price"),
        col("quantity")
    )
)


# ============================================================
# MOST IMPORTANT THINGS TO REMEMBER
# ============================================================

# 1. UDF = User Defined Function.
#
# 2. It allows us to use our own Python function on
#    Spark DataFrame columns.
#
# 3. Create a normal Python function first.
#
# 4. Convert it to a UDF using:
#
#    udf(function, return_type)
#
# 5. Use the UDF with a DataFrame column:
#
#    df.withColumn(
#        "new_column",
#        my_udf(col("column"))
#    )
#
# 6. Always specify the UDF return type.
#
# 7. Handle None if the input column can contain NULL values.
#
# 8. Prefer Spark's built-in functions whenever possible.
#
# 9. Python UDFs can be slower because Spark has to execute
#    Python code instead of using its optimized built-in
#    execution path.
#
# 10. UDF should generally be used when built-in Spark
#     functions cannot conveniently solve the problem.